# Assignment 5 - Evaluation metrics

RAGAS Metrics: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

## LLM As Judge
1. [Noise sensitivity](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/noise_sensitivity/) - % incorrect claims out of total claims
  - Response
  - Context
2. [Faithfulness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/) - % of claims supported by retrieved context
  - Response
  - Context
3. Persona preferences - Does the response reflect the preferences of the persona? [0,1,2] scale
  - Response
  - Persona description
  - Gold response example (multi-shot)
  - report as normalized score [0,1]
4. [Answer relevancy](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/#answer-relevancy) - relevancy of the Gold question to the generated response
  - Gold question
  - Generated response
  - Judge generates 3 questions based on the response
  - Average cosine similarity of generated questions to the gold question


## Mathematical Eval
5. [BERT Score](https://arxiv.org/abs/1904.09675) - compares embeddings of gold asnwer to generates response
  - Gold answer
  - Generated response
6. [Honesty/Hallucination](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/agents/#topic-adherence) - Does the LLM return the "I don't know" phrase [IDK] when it cannot find relevant context (F1 score).
  - Precision = (IDK & NO Context) / (IDK & Have Context + IDK & NO Context)
  - Recall = (IDK & NO Context) / (Factual response & NO Context + IDK Response & NO Context)
  - F1 Score = 2*Precision*Recall / (Precision + Recall)


===========================================================================================================

## 1. Setup

We will first install a number of libraries and import what we will need.





In [2]:
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia


In [4]:
import os
import numpy as np
import time
import locale

# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

COHERE_API_KEY = userdata.get('COHERE_API_KEY')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

locale.getpreferredencoding = lambda: "UTF-8"


/tmp/ipykernel_501/1138715678.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import ArxivLoader


In [5]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers.utils import get_json_schema
from typing import Annotated

In [26]:
%%capture
!pip install -U sentence_transformers
from sentence_transformers import CrossEncoder


!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_50.tar.gz" -C /content

In [27]:
%%capture

class VectorStoreRetriever():
      EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
      def __init__(self):
          self.collection_name = "rag_tech_db_250"
          path = '/content/qdrant_storage'
          self.init_embeddings(self.EMBEDDINGS_MODEL)
          self.init_vector_store(path, self.collection_name)
      def init_vector_store(self, path, collection_name):
          self.vector_store = QdrantVectorStore(
              client=QdrantClient(path=path),
              embedding=self.base_embeddings,
              collection_name=collection_name,
              distance=Distance.DOT)
      def init_embeddings(self, embeddings_model):
          self.base_embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)


vector_store = VectorStoreRetriever().vector_store

In [ ]:
%%capture

class CrossEncoderWrapper():
    def __init__(self, threshold, doc_limit):
        self.threshold = threshold
        self.doc_limit = doc_limit
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", activation_fn=torch.nn.Sigmoid())
    def scores(self, query, context):
        return self.cross_encoder.predict([(query, doc.page_content) for doc in context])
    def filtered(self, query, context):
        scores = self.scores(query, context)
        reranked_docs = [(score, d) for score, d in sorted(zip(scores, context), key=lambda x: x[0], reverse=True)]
        return list(filter(lambda reranked: reranked[0] >= self.threshold, reranked_docs))[:self.doc_limit]

cross_encoder = CrossEncoderWrapper(threshold=0.5, doc_limit=5)


In [6]:
# LOAD QWEN 3 LLM
# judge for Gold Answer generation
qwen_model_name = "Qwen/Qwen3-8B"

qwen_quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

# load the tokenizer and the model
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float32,
    device_map="auto",
    max_length = None,
    quantization_config=qwen_quantization_config
)
qwen_model.config.pad_token_id = qwen_model.config.eos_token_id

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens=1000,
    temperature=0.3,
    top_p=0.5,
    do_sample=True,
    repetition_penalty=1.2
)

qwen_llm = HuggingFacePipeline(pipeline=qwen_pipe)
qwen_chat = ChatHuggingFace(llm=qwen_llm)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'repetition_penalty', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Noise sensitivity
https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/noise_sensitivity/

Definition: % incorrect claims out of total claims

Required data:
  - Response
  - Context

## Faithfulness
https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/

Definition: % of claims supported by retrieved context

- Response
- Context

In [48]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
import re
import json

class ClaimsEvaluator():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def evaluate(self, response: str, context: str):
        content = self.TEMPLATE.format(
            context=context,
            response=response
        )

        message = [
            {"role": "user", "content": content}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.qwen_extract_and_parse_json(decoded_output)

    def qwen_extract_and_parse_json(self, text_output):
        text_output = text_output.split("</think>")[1].strip()
        json_output = re.search(r"```(?:json)?\s*\n(.*?)\s*\n```", text_output, re.DOTALL).group(1)
        return json.loads(json_output)

    TEMPLATE = """Your task is to evaluate the claims of a provided statement based ONLY on provided context.

A claim is a single, simplified and separable clause that can be evaluated independently.
You must break the statement into its constituent claims independently of the context.

After the statement is broken into claims, classify each in exactly 1 of 3 possible categories: faithful, noisy, or irrelevant.
A claim is faithful if it is supported or verified by the supplied context. The factual basis of a faithful claim MUST appear in the context.
A claim is noisy if it is incorrect, factually wrong, or contrary to the information in the context.
Claims that are not noisy and not faithful are irrelevant - the context does not mention the information in the claim, so it cannot be determined as noisy or faithful to the context.

Context:
Frogs are green.

Statement:
Darryl is a frog, therefore he is green. Frogs are brown.

Answer:
{{
"claims": ["Darryl is a frog", "he is green", "Frogs are brown"],
"claims_count": 3,
"noisy_count": 1,
"faithful_count": 1,
"irrelevant_count": 1
}}

Context:
{context}

Statement:
{response}

Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"claims": strings[array of factual claims],
"claims_count": integer (total count of all claims),
"noisy_count": integer (count of noisy claims),
"faithful_count": integer (count of faithful claims),
"irrelevant_count": integer (count of irrelevant claims)
}}
```"""

claims_evaluator = ClaimsEvaluator(qwen_tokenizer, qwen_model)


In [51]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
import re
import json

class PersonaEvaluator():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def evaluate(self, response: str, persona: str):
        content = self.TEMPLATE.format(
            persona=persona,
            response=response
        )

        message = [
            {"role": "user", "content": content}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=True,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output):
        text_output = text_output.split("</think>")[1].strip()
        return text_outpu)

    TEMPLATE = """Your task is to evaluate how well a statement reflects the communication preferences of the user.
Some users like detailed and highly technical answers to their questions, while other users like high-level and simplified respones.

Preferences:
{persona}

Statement:
{response}

Your evaluation must be an integer in [0, 1, 2]
Evaluation:"""

persona_evaluator = PersonaEvaluator(qwen_tokenizer, qwen_model)


SyntaxError: unmatched ')' (371785191.py, line 35)

In [50]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
import re
import json

class RelevancyEvaluator():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def evaluate(self, response: str):
        content = self.TEMPLATE.format(
            response=response
        )

        message = [
            {"role": "user", "content": content}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=True,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280, temperature=0.9)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output):
        text_output = text_output.split("</think>")[1].strip()
        return text_output

    TEMPLATE = """You are test writing specialist. You will be given a statement that is the intended answer to a question.
Your task is to write a question based ONLY on the content of the statement.
When paired together, the statement should be a reasonable and accurate answer to the question.

Statement:
{response}

Question:"""

relevancy_evaluator = RelevancyEvaluator(qwen_tokenizer, qwen_model)


In [54]:
class NaiveRAG():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def invoke(self, question: str, context: str): # Added context parameter
        content = self.TEMPLATE.format(
            context=context,
            question=question
        )
        message = [
            {"role": "user", "content": content}
        ]
        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output: str):
        assistant_tag = "assistant"
        if assistant_tag in text_output:
            response_part = text_output.split(assistant_tag, 1)[1].strip()
            if "</think>" in response_part:
                return response_part.split("</think>", 1)[1].strip()
            return response_part

        return text_output.strip()


    TEMPLATE = """You are an expert in generative AI. Your task is to answer the question based ONLY on the provided context.

Context:
{context}

Question:
{question}

Answer:"""


rag = NaiveRAG(qwen_tokenizer, qwen_model)

In [53]:

research_persona = """This user is an engineer, who requires detailed technical information when they ask questions.
You will help them by answering about generative AI concepts, internal system architecture, and implementation details."""

marketing_persona = """This user is a marketer who will ask questions about generative AI in order to better understand the products and the field as a whole.
They prefer high level answers that explain concept over technical detail.
You will help them find accurate, approved messaging about generative AI features, competitive positioning, and technical capabilities to accelerate content production."""


test_qs = {"7": {
    "question": "What are some examples of tasks that involve evaluating the robustness of generative AI models?",
    "question_id": 7,
    "context_id": "e196c8cfb0a14c70bb03bb0b370bd0bd",
    "persona": research_persona,
    "metadata": {
      "producer": "pdfTeX-1.40.25",
      "creator": "LaTeX with hyperref",
      "creationdate": "2024-03-28T00:54:45+00:00",
      "source": "https://arxiv.org/pdf/2312.10997.pdf",
      "file_path": "https://arxiv.org/pdf/2312.10997.pdf",
      "total_pages": 21,
      "format": "PDF 1.5",
      "title": "",
      "author": "",
      "subject": "",
      "keywords": "",
      "moddate": "2024-03-28T00:54:45+00:00",
      "trapped": "",
      "modDate": "D:20240328005445Z",
      "creationDate": "D:20240328005445Z",
      "page": 12,
      "page_num": 12,
      "doc_source": "ArXiv",
      "id": "2312.10997",
      "split_id": 68,
      "doc_num": 29
    }
  },
  "8": {
    "question": "How does incorporating task instructions during encoding enhance the versatility of the INSTRUCTOR model for different language tasks?",
    "question_id": 8,
    "context_id": "4f27ed5307dc4aa682291d306299feac",
    "persona": marketing_persona,
    "metadata": {
      "producer": "pdfTeX-1.40.25",
      "creator": "LaTeX with hyperref",
      "creationdate": "2023-05-31T00:45:57+00:00",
      "source": "https://arxiv.org/pdf/2212.09741.pdf",
      "file_path": "https://arxiv.org/pdf/2212.09741.pdf",
      "total_pages": 18,
      "format": "PDF 1.5",
      "title": "",
      "author": "",
      "subject": "",
      "keywords": "",
      "moddate": "2023-05-31T00:45:57+00:00",
      "trapped": "",
      "modDate": "D:20230531004557Z",
      "creationDate": "D:20230531004557Z",
      "page": 1,
      "page_num": 1,
      "doc_source": "ArXiv",
      "id": "2212.09741",
      "split_id": 9,
      "doc_num": 14
    }
  }
}

for q in test_qs.values():

    print("*" * 60)
    question = q['question']
    print(f"Question: {question}")
    context_id = q['context_id']

    context = vector_store.get_by_ids([context_id])

    response = rag.invoke(question=question, context=context)
    print()
    print(f"Response: {response}")

    print("=== CLAIMS ===")
    claims_evaluation = claims_evaluator.evaluate(response=response, context=context)
    print(claims_evaluation)
    print()
    print("=== PERSONA ===")
    persona = q['persona']
    persona_evaluation = persona_evaluator.evaluate(response=response, persona=persona)
    print(persona_evaluation)
    print()
    print("=== RELEVANCY ===")
    for n in range(3):
      relevancy_evaluation = relevancy_evaluator.evaluate(response=response)
      print(f"{n+1}. {relevancy_evaluation}")
    print()



************************************************************
Question: What are some examples of tasks that involve evaluating the robustness of generative AI models?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Response: user
You are an expert in generative AI. Your task is to answer the question based ONLY on the provided context.

Context:
[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-03-28T00:54:45+00:00', 'source': 'https://arxiv.org/pdf/2312.10997.pdf', 'file_path': 'https://arxiv.org/pdf/2312.10997.pdf', 'total_pages': 21, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-28T00:54:45+00:00', 'trapped': '', 'modDate': 'D:20240328005445Z', 'creationDate': 'D:20240328005445Z', 'page': 12, 'page_num': 12, 'doc_source': 'ArXiv', 'id': '2312.10997', 'split_id': 68, 'doc_num': 29, '_id': 'e196c8cfb0a14c70bb03bb0b370bd0bd', '_collection_name': 'rag_tech_db_250'}, page_content='Language Modeling\nWikiText-103 [147]\n[5], [29], [64], [71]\nStrategyQA [148]\n[14], [24], [48], [51], [55], [58]\nFact Checking/Verification\nFEVER [149]\n[4], [13], [27], [34], [42], [50]\nPubHealth [150]\n[25]

JSONDecodeError: Expecting value: line 2 column 11 (char 12)